In [1]:
import sys
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.inventory.policy import (
    calculate_target_inventory,
    calculate_recommended_order_qty,
)

In [2]:
from src.inventory.simulation import evaluate_policy
from src.inventory.simulation import (create_purchase_order, calculate_inventory_position)
from src.inventory.simulation import evaluate_reorder_decision
from src.inventory.simulation import create_policy_order
from src.inventory.simulation import simulate_day_with_policy
from src.inventory.simulation import simulate_inventory_with_policy

In [3]:
total_forecast = 300
safety_stock = 50

target_inventory = calculate_target_inventory(
    total_forecast,
    safety_stock,
)

print("Target inventory:", target_inventory)

Target inventory: 350


In [4]:
inventory_position = 200
reorder_required = True

order_qty = calculate_recommended_order_qty(
    target_inventory=target_inventory,
    inventory_position=inventory_position,
    reorder_required=reorder_required,
)

print("Recommended order quantity:", order_qty)

Recommended order quantity: 150


In [5]:
order_qty = calculate_recommended_order_qty(
    target_inventory=target_inventory,
    inventory_position=400,
    reorder_required=False,
)

print("Recommended order quantity:", order_qty)

Recommended order quantity: 0


In [6]:
orders = [
    create_purchase_order(
        order_date="2025-01-05",
        quantity=100,
        lead_time_days=2,
    ),
    create_purchase_order(
        order_date="2025-01-08",
        quantity=50,
        lead_time_days=12,
    ),
]

In [7]:
inventory_position = calculate_inventory_position(
    current_stock=80,
    purchase_orders=orders,
    current_date="2025-01-10",
)

print(inventory_position)

130


In [8]:
reorder_required = evaluate_reorder_decision(
    inventory_position=110,
    lead_time_demand=100,
    safety_stock=30,
)

print(reorder_required)

True


In [9]:
reorder_required = evaluate_reorder_decision(
    inventory_position=150,
    lead_time_demand=100,
    safety_stock=30,
)

print(reorder_required)

False


In [10]:
order = create_policy_order(
    date="2025-01-10",
    total_forecast=300,
    lead_time_demand=100,
    safety_stock=30,
    inventory_position=110,
    lead_time_days=4,
)

print(order)

PurchaseOrder(order_date=Timestamp('2025-01-10 00:00:00'), arrival_date=Timestamp('2025-01-14 00:00:00'), quantity=220)


In [11]:
purchase_orders = []

daily_result, new_order = simulate_day_with_policy(
    date="2025-01-10",
    opening_stock=100,
    demand=30,
    total_forecast=300,
    lead_time_demand=100,
    safety_stock=30,
    purchase_orders=purchase_orders,
    lead_time_days=4,
)

print(daily_result)
print(new_order)

{'date': Timestamp('2025-01-10 00:00:00'), 'opening_stock': 100, 'arrival_qty': 0, 'demand': 30, 'units_fulfilled': 30, 'stockout_units': 0, 'closing_stock': 70, 'inventory_position': 70, 'order_qty': 260, 'order_date': Timestamp('2025-01-10 00:00:00'), 'arrival_date': Timestamp('2025-01-14 00:00:00')}
PurchaseOrder(order_date=Timestamp('2025-01-10 00:00:00'), arrival_date=Timestamp('2025-01-14 00:00:00'), quantity=260)


In [12]:
test_demand = pd.DataFrame(
    {
        "date": pd.date_range("2025-01-01", periods=5),
        "units_sold": [30, 40, 50, 20, 10],
    }
)

simulation = simulate_inventory_with_policy(
    demand_df=test_demand,
    initial_stock=100,
    total_forecast=300,
    lead_time_demand=100,
    safety_stock=30,
    lead_time_days=2,
)

simulation

,date,opening_stock,arrival_qty,demand,units_fulfilled,stockout_units,closing_stock,inventory_position,order_qty,order_date,arrival_date
0,2025-01-01,100,0,30,30,0,70,70,260,2025-01-01,2025-01-03
1,2025-01-02,70,0,40,40,0,30,290,0,NaT,NaT
2,2025-01-03,30,260,50,50,0,240,240,0,NaT,NaT
3,2025-01-04,240,0,20,20,0,220,220,0,NaT,NaT
4,2025-01-05,220,0,10,10,0,210,210,0,NaT,NaT


In [13]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

sales = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "sales.csv"
)

sales.head()

,date,product_id,product_name,category,price,discount,promotion,day_of_week,month,is_weekend,units_sold
0,2024-01-01,P001,Wireless Headphones,Electronics,1999.0,0,0,0,1,False,33
1,2024-01-02,P001,Wireless Headphones,Electronics,1999.0,0,0,1,1,False,39
2,2024-01-03,P001,Wireless Headphones,Electronics,1999.0,0,0,2,1,False,39
3,2024-01-04,P001,Wireless Headphones,Electronics,1799.1,10,1,3,1,False,59
4,2024-01-05,P001,Wireless Headphones,Electronics,1999.0,0,0,4,1,False,35


In [14]:
sales["product_id"].unique()

<StringArray>
['P001', 'P002', 'P003', 'P004', 'P005']
Length: 5, dtype: str

In [15]:
product_id = sales["product_id"].iloc[0]

product_sales = (
    sales[sales["product_id"] == product_id]
    .copy()
)

product_sales["date"] = pd.to_datetime(
    product_sales["date"]
)

product_sales = (
    product_sales
    .sort_values("date")
    .reset_index(drop=True)
)

product_sales[
    ["date", "product_id", "units_sold"]
].head(10)

,date,product_id,units_sold
0,2024-01-01,P001,33
1,2024-01-02,P001,39
2,2024-01-03,P001,39
3,2024-01-04,P001,59
4,2024-01-05,P001,35
5,2024-01-06,P001,41
6,2024-01-07,P001,30
7,2024-01-08,P001,42
8,2024-01-09,P001,43
9,2024-01-10,P001,34


In [16]:
test_period = product_sales.head(30)[
    ["date", "units_sold"]
].copy()

test_period.head()

,date,units_sold
0,2024-01-01,33
1,2024-01-02,39
2,2024-01-03,39
3,2024-01-04,59
4,2024-01-05,35


In [17]:
simulation = simulate_inventory_with_policy(
    demand_df=test_period,
    initial_stock=100,
    total_forecast=300,
    lead_time_demand=100,
    safety_stock=30,
    lead_time_days=4,
)

simulation

,date,opening_stock,arrival_qty,demand,units_fulfilled,stockout_units,closing_stock,inventory_position,order_qty,order_date,arrival_date
0,2024-01-01,100,0,33,33,0,67,67,263,2024-01-01,2024-01-05
1,2024-01-02,67,0,39,39,0,28,291,0,NaT,NaT
2,2024-01-03,28,0,39,28,11,0,263,0,NaT,NaT
3,2024-01-04,0,0,59,0,59,0,263,0,NaT,NaT
4,2024-01-05,0,263,35,35,0,228,228,0,NaT,NaT
5,2024-01-06,228,0,41,41,0,187,187,0,NaT,NaT
6,2024-01-07,187,0,30,30,0,157,157,0,NaT,NaT
7,2024-01-08,157,0,42,42,0,115,115,215,2024-01-08,2024-01-12
8,2024-01-09,115,0,43,43,0,72,287,0,NaT,NaT
9,2024-01-10,72,0,34,34,0,38,253,0,NaT,NaT
